# L10: 工具定义与注册

> 理解 Agent 的「手」——如何让 LLM 能够调用外部工具

In [1]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
from pathlib import Path

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

# 验证模块导入
try:
    from backend.tools import ToolRegistry, register_tool, Tool
    print("✓ Tools 模块导入成功")
except ImportError as e:
    print(f"✗ 模块导入失败: {e}")

项目路径: c:\Users\zying\Desktop\pro_agent\agent-learning-project
Python 版本: 3.14.4
✗ 模块导入失败: cannot import name 'Tool' from 'backend.tools' (c:\Users\zying\Desktop\pro_agent\agent-learning-project\backend\tools\__init__.py)


## 10.1 什么是工具 (Tool)？

### 工具的本质

**工具** = LLM 可以调用的「外部函数」，让 Agent 能够：
- 获取实时信息（天气、股票、新闻）
- 执行操作（计算器、文件操作、API 调用）
- 访问知识库（搜索、数据库查询）

### Tool Use 的两种实现方式

| 方式 | 说明 | 例子 |
|------|------|------|
| **Native Function Calling** | LLM 原生支持，直接返回结构化调用 | OpenAI、Anthropic、DeepSeek |
| **Prompt-based Tool Use** | 通过 Prompt 让 LLM 输出工具调用格式 | 需要手动解析 |

### 工具调用的完整流程

```
用户问题
   ↓
LLM 分析 → 选择工具 → 生成调用参数
   ↓
执行工具 → 获取结果
   ↓
将结果返回给 LLM → 生成最终回答
```

## 10.2 项目中的工具系统架构

### 设计模式：装饰器注册 + 工厂模式

```python
# 1. 定义工具（使用装饰器自动注册）
@register_tool
class MyTool(Tool):
    name = "my_tool"
    description = "做什么的"
    
    async def execute(self, **kwargs):
        return "结果"

# 2. 获取工具（通过注册表）
tool = ToolRegistry.get("my_tool")
result = await tool.execute(arg1="value1")
```

In [2]:
# 查看项目中已注册的工具
from backend.tools.registry import ToolRegistry

print("=== 已注册的工具 ===
")
tools = ToolRegistry.get_all_tools()
for name, tool in tools.items():
    print(f"📦 {name}")
    print(f"   描述: {tool.description}")
    # parameters 是列表，获取参数名称
    param_names = [p.name for p in tool.parameters] if tool.parameters else []
    print(f"   参数: {param_names}")
    print()


=== 已注册的工具 ===

📦 search
   描述: 在互联网上搜索信息（优先使用维基百科，百科类信息更准确）


AttributeError: 'list' object has no attribute 'keys'

## 10.3 工具基类：Tool

### 查看 Tool 基类定义

In [ ]:
import inspect
from backend.tools.base import Tool

print("=== Tool 基类 ===\n")
print(inspect.getsource(Tool))

### Tool 的核心属性

| 属性 | 类型 | 说明 |
|------|------|------|
| `name` | str | 工具唯一标识符 |
| `description` | str | 工具功能描述（LLM 会看到） |
| `parameters` | dict | 参数定义（JSON Schema 格式） |
| `execute()` | method | 实际执行逻辑（异步） |

### 为什么 description 很重要？

LLM 通过 **description** 决定何时调用哪个工具。好的描述应该：
- 清晰说明工具的功能
- 说明适用场景
- 提及参数要求

## 10.4 装饰器注册机制

### @register_tool 的工作原理

In [ ]:
# 查看 register_tool 装饰器的实现
import inspect
from backend.tools.registry import register_tool

print("=== register_tool 装饰器源码 ===\n")
print(inspect.getsource(register_tool))

### 装饰器注册的好处

| 好处 | 说明 |
|------|------|
| **零侵入** | 不需要在主代码中手动注册 |
| **自动发现** | 导入模块时自动注册 |
| **解耦** | 工具定义与注册分离 |
| **易于扩展** | 添加新工具只需定义类 |

## 10.5 练习：创建自定义工具

### 练习 1：创建一个简单的问候工具

In [ ]:
from backend.tools import Tool, register_tool
from typing import Optional

@register_tool
class GreetingTool(Tool):
    """问候工具示例"""
    
    name = "greeting"
    description = "向指定的人打招呼"
    
    parameters = {
        "name": {
            "type": "string",
            "description": "要问候的人名",
            "required": True
        },
        "language": {
            "type": "string",
            "description": "问候语言 (zh/en/jp)",
            "required": False,
            "default": "zh"
        }
    }
    
    async def execute(self, name: str, language: str = "zh") -> str:
        """执行问候"""
        greetings = {
            "zh": f"你好，{name}！",
            "en": f"Hello, {name}!",
            "jp": f"こんにちは、{name}！"
        }
        return greetings.get(language, greetings["zh"])

# 测试工具
async def test_greeting():
    tool = ToolRegistry.get("greeting")
    print(await tool.execute(name="小明", language="zh"))
    print(await tool.execute(name="Alice", language="en"))

await test_greeting()

### 练习 2：创建一个获取当前时间的工具

In [ ]:
from datetime import datetime

@register_tool
class GetCurrentTimeTool(Tool):
    """获取当前时间工具"""
    
    name = "get_current_time"
    description = "获取当前的日期和时间，支持指定时区"
    
    parameters = {
        "timezone": {
            "type": "string",
            "description": "时区（如 Asia/Shanghai）",
            "required": False,
            "default": "Asia/Shanghai"
        },
        "format": {
            "type": "string",
            "description": "时间格式 (full/date/time)",
            "required": False,
            "default": "full"
        }
    }
    
    async def execute(self, timezone: str = "Asia/Shanghai", format: str = "full") -> str:
        """获取当前时间"""
        try:
            # 简化实现，实际应使用 pytz 处理时区
            now = datetime.now()
            
            if format == "date":
                return now.strftime("%Y-%m-%d")
            elif format == "time":
                return now.strftime("%H:%M:%S")
            else:
                return now.strftime("%Y-%m-%d %H:%M:%S")
        except Exception as e:
            return f"Error: {str(e)}"

# 测试
async def test_time():
    tool = ToolRegistry.get("get_current_time")
    print(f"完整时间: {await tool.execute()}")
    print(f"仅日期: {await tool.execute(format='date')}")
    print(f"仅时间: {await tool.execute(format='time')}")

await test_time()

## 10.6 工具注册表：ToolRegistry

### 注册表的核心方法

In [ ]:
# 查看 ToolRegistry 的方法
from backend.tools.registry import ToolRegistry

print("=== ToolRegistry 公共方法 ===\n")
for name in dir(ToolRegistry):
    if not name.startswith('_'):
        attr = getattr(ToolRegistry, name)
        if callable(attr):
            print(f"  - {name}")

### ToolRegistry 使用示例

In [ ]:
import asyncio

async def registry_demo():
    """演示注册表的使用"""
    
    # 1. 列出所有工具
    print("=== 1. 所有已注册工具 ===")
    tools = ToolRegistry.get_all_tools()
    for name in tools:
        print(f"  - {name}")
    
    # 2. 获取单个工具
    print("\n=== 2. 获取特定工具 ===")
    calculator = ToolRegistry.get("calculator")
    print(f"Calculator 工具描述: {calculator.description}")
    
    # 3. 检查工具是否存在
    print("\n=== 3. 检查工具是否存在 ===")
    print(f"calculator 存在: {ToolRegistry.has('calculator')}")
    print(f"unknown_tool 存在: {ToolRegistry.has('unknown_tool')}")
    
    # 4. 获取工具的 JSON Schema（用于 Function Calling）
    print("\n=== 4. 工具 JSON Schema ===")
    schema = calculator.to_json_schema()
    print(json.dumps(schema, indent=2, ensure_ascii=False))
    
    # 5. 执行工具
    print("\n=== 5. 执行工具 ===")
    result = await calculator.execute(expression="2 + 2")
    print(f"2 + 2 = {result}")

await registry_demo()

## 10.7 工具设计最佳实践

### 命名规范

```
好的命名:                    不好的命名:
- search_web                - tool1
- calculate_expression      - do_something
- get_weather               - helper
- send_email                - process
```

### Description 写法

| 差 | 中 | 好 |
|----|----|-----|
| 计算器 | 进行数学计算 | 执行数学表达式计算，支持加减乘除和括号 |
| 搜索工具 | 在网上搜索 | 根据关键词搜索网络信息，返回相关结果 |
| 天气 | 获取天气信息 | 查询指定城市的当前天气和未来预报 |

### 参数设计

- **必要参数** 设为 `required: True`
- 提供 **默认值** 减少必需参数
- 使用 **清晰类型** (string/number/boolean)
- **添加验证** 在 execute 方法中

## 10.8 常见面试问题

**Q: 为什么要用装饰器注册工具？**
- A:
  - 零侵入：工具定义后自动注册
  - 解耦：工具不需要知道注册表的存在
  - 易扩展：新增工具只需定义类 + 加装饰器
  - 可发现：导入模块时自动扫描注册

**Q: Tool 的 execute 方法为什么要用 async？**
- A:
  - 工具可能涉及 I/O 操作（API 调用、文件读写）
  - async 可以避免阻塞事件循环
  - 与 LLM 调用保持一致的异步风格
  - 支持多工具并发执行

**Q: 如何让 LLM 知道有哪些工具可用？**
- A:
  1. 在 system prompt 中列出工具
  2. 使用 Function Calling API（OpenAI/Anthropic）
  3. 在 few-shot 示例中展示工具调用
  4. 通过 to_json_schema() 生成工具描述

**Q: 工具执行失败时如何处理？**
- A:
  - execute 方法捕获异常并返回错误信息
  - 使用 try-except 包裹工具调用
  - 将错误信息返回给 LLM，让它尝试其他方法
  - 可以设置超时和重试机制

**Q: 如何实现工具的权限控制？**
- A:
  - 在 Tool 基类中添加 permission_level 属性
  - 在 execute 前检查权限
  - 将敏感操作（文件删除、邮件发送）标记为高危
  - 可以要求用户确认高危操作

---

## 总结

本课程学习了工具系统的核心知识：

| 知识点 | 说明 |
|--------|------|
| **Tool 概念** | LLM 调用外部函数的能力 |
| **Tool 基类** | name、description、parameters、execute |
| **装饰器注册** | @register_tool 自动注册到注册表 |
| **ToolRegistry** | 工具的中央管理器 |
| **JSON Schema** | 用于 Function Calling 的工具描述 |
| **最佳实践** | 清晰命名、详细描述、参数验证 |

**下一步**: L11 将深入讲解工具参数解析与验证！